In [35]:
from datasets import load_dataset
from transformers import MT5Tokenizer, MT5Model
import torch
import math
from tqdm import tqdm
from sklearn.cluster import MiniBatchKMeans


In [36]:
ds_stream = load_dataset("HiTZ/euscrawl", split='train[:10000]')

KeyboardInterrupt: 

In [ ]:
ds_stream['plain_text'][0].split('\n')

['Pat Garrett and Billy the Kid',
 'Pat Garrett and Billy the Kid (euskaraz "Pat Garrett eta Billy Haurra") 1973ko Sam Peckinpah estatubatuar zinema zuzendariren western filma bat da. Bertan, James Coburn, Kris Kristofferson, Bob Dylan, Slim Pickens eta Jason Robards aktoreek antzeztu zuten.',
 'Filma honetako soinu banda Bob Dylan kantautorearen "Pat Garrett &amp; Billy the Kid" izeneko diskoan argitaratu zen, bertan eta filmean kantautore honen "Knockin\' on Heaven\'s Door" ("Zeruko Atean Joka") abesti ospetsua entzun daitekeelarik.',
 'Argumentua.',
 'Denek William Bonney (Kris Kristofferson) pistolariaera "Billy Haurra" ezizenez ezagutzen dute. Epaiketa ondoren urkamendian hiltzera zigortua izan ondoren, zigorraren zain Lincoln herriko kartzelan espetxeratua dago. Baina ustekabean bere eskuetara colt 44 pistola bat iritsiko da, harekin bere zaindariak hil eta Mexikora ihes egingo du. Halere, beste garai batean bere adiskide izan zen Pat Garrett (James Coburn) sheriffa hura atzemate

In [ ]:
with open("./sentences.txt", 'w') as f:
    for datum in ds_stream:        
        f.write( datum['plain_text'])
        f.write("\n")

    

In [ ]:
# Load pre-trained MT5 model and tokenizer
model_name = 'google/mt5-small'
tokenizer = MT5Tokenizer.from_pretrained(model_name)
model = MT5Model.from_pretrained(model_name, device_map='auto')


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/gaueko0/users/asalem/anaconda3/envs/rag-llama/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
def encode_sentence(sentence):
    
    # Encode a sentence
    encoded_input = tokenizer(sentence, return_tensors='pt').to(model.device)
    # Get the model's output
    output = model.encoder(
    input_ids=encoded_input["input_ids"], 
    attention_mask=encoded_input["attention_mask"], 
    return_dict=True)
    # Access the hidden states
    last_hidden_state = output.last_hidden_state
    return last_hidden_state.detach().cpu().numpy().squeeze()[0,:]




In [41]:

with open("./sentences.txt", 'r') as f:
    
    
    kmeans = MiniBatchKMeans(n_clusters=5, random_state=0, batch_size=100, n_init="auto")
        
    lines = f.readlines()
    
    batches = math.ceil(len(lines)/100)
    
    for batch_id in tqdm(range(batches)):
        
        batch_start = batch_id*100
        batch_end = min(len(lines), batch_start+100)
        batch = lines[batch_start:batch_end]
        batch = filter(lambda x: len(x.strip())>1, batch)
        encodings = [ encode_sentence(line) for line in batch ]

        kmeans = kmeans.partial_fit(encodings)
    
    

100%|██████████| 639/639 [17:36<00:00,  1.65s/it]


In [47]:
kmeans.cluster_centers_ = kmeans.cluster_centers_.astype(float)


In [54]:
with open("./sentences.txt", 'r') as f:

    all_res = []
    for idx, line in tqdm(enumerate(f.readlines()), total=63803):
        try:
            encoding = encode_sentence(line)
            out = kmeans.predict( encoding.reshape(1,-1).astype(float) )
            all_res.append((idx, out[0]))
        except:
            ...

  0%|          | 11/63803 [00:00<10:22, 102.48it/s]

100%|██████████| 63803/63803 [10:12<00:00, 104.10it/s]


In [56]:
import pandas as pd

In [57]:
clusters = pd.DataFrame.from_records(all_res, columns=['sentence_idx', 'cluster'])


In [59]:
clusters.to_csv("./clusters_ids.csv", index=0)

In [60]:
sentences = [ (idx,line) for idx, line in enumerate(open('./sentences.txt', 'r').readlines())]

In [61]:
sentences_df = pd.DataFrame.from_records(sentences, columns=['sentence_idx', 'text'])


In [64]:
sentences_df.merge(clusters, on='sentence_idx').to_csv('./sentences_clustered.csv', index=False)

In [66]:
df  = pd.read_csv("./sentences_clustered.csv", header=0)
df

,sentence_idx,text,cluster
0,0,Pat Garrett and Billy the Kid\n,1
1,1,"Pat Garrett and Billy the Kid (euskaraz ""Pat G...",1
2,2,Filma honetako soinu banda Bob Dylan kantautor...,2
3,3,Argumentua.\n,3
4,4,Denek William Bonney (Kris Kristofferson) pist...,1
...,...,...,...
63796,63798,Urriza eta Barrenetxea IV.a txapeldun\n,1
63797,63799,40-31 irabazi diete Ezkurra eta Olazarri Orona...,3
63798,63800,Javier Urrizak eta Endika Barrenetxeak jantzi ...,4
63799,63801,Urriza eta Barrenetxea IV.a indartsu hasi dira...,1


In [67]:
df.cluster.value_counts()

cluster
1    16203
3    15779
4    11912
2    11770
0     8137
Name: count, dtype: int64

In [68]:
cluster_dict = df.groupby('cluster')['sentence_idx'].apply(list).to_dict()

In [73]:
import numpy as np
def select_random_sentences(current_sentence_idx):
    current_cluster = df.loc[df['sentence_idx'] == current_sentence_idx, 'cluster'].values[0]
    other_sentences_in_cluster = cluster_dict[current_cluster].copy()
    other_sentences_in_cluster.remove(current_sentence_idx)
    
    selected_indices = np.random.choice(other_sentences_in_cluster, 1, replace=False)
    
    return selected_indices

In [74]:
selected_sentences = select_random_sentences(3)


In [78]:
df['target_sentence'] = df['sentence_idx'].apply(lambda x: select_random_sentences(x).item())

In [83]:
df['target_text'] = df.apply(lambda x: df.loc[df['sentence_idx'] == x['target_sentence'], 'text'].values[0], axis=1)

In [84]:
df.head(5)

,sentence_idx,text,cluster,target_sentence,target_text
0,0,Pat Garrett and Billy the Kid\n,1,24899,Lezoko Udaleko Ingurumen eta Trantsizio Ekolog...
1,1,"Pat Garrett and Billy the Kid (euskaraz ""Pat G...",1,33533,"2013ko urtarrilaren 16an, GitHub-ek adierazi z..."
2,2,Filma honetako soinu banda Bob Dylan kantautor...,2,43359,Etxebizitzen alokairurako eta sustapen ekonomi...
3,3,Argumentua.\n,3,34187,Historia.\n
4,4,Denek William Bonney (Kris Kristofferson) pist...,1,40270,"Dostoievski 1863an hasi zen jokoan, Wiesbadene..."


In [85]:
df.to_csv('./sentences_clustered.csv', index=False)